In [2]:
import json
import duckdb
import numpy as np
import pandas as pd
print("✔ Libraries imported successfully.")

from datasets import load_dataset
print("✔ Hugging Face datasets library loaded.")

print("Starting download from Hugging Face (this should take ~15 seconds)...")
train_ds = load_dataset('Hacker0x01/disclosed_reports', split="train").to_pandas()
print("✔ Train split downloaded.")

test_ds = load_dataset('Hacker0x01/disclosed_reports', split='test').to_pandas()
print("✔ Test split downloaded.")

validate_ds = load_dataset('Hacker0x01/disclosed_reports', split='validation').to_pandas()
print("✔ Validation split downloaded.")

df = pd.concat([train_ds, test_ds, validate_ds], ignore_index=True)
print(f"✔ Combined dataset shape: {df.shape}")

✔ Libraries imported successfully.


/Users/adamghriss/Desktop/duckdb_pipeline/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✔ Hugging Face datasets library loaded.
Starting download from Hugging Face (this should take ~15 seconds)...


Generating validation split: 100%|██████████| 1262/1262 [00:00<00:00, 114095.05 examples/s]


✔ Train split downloaded.
✔ Test split downloaded.
✔ Validation split downloaded.
✔ Combined dataset shape: (12618, 14)


In [3]:

# --- Finalizing Phase 2: Processing & Saving the Bronze Layer ---
print("Processing nested fields and casting dates...")

# 1. Handle nested dictionaries to prevent Parquet schema/serialization errors
for col in ['reporter', 'team', 'weakness', 'structured_scope']:
    df[col] = df[col].apply(lambda x: x if isinstance(x, dict) else {})
    df[col + "_json"] = df[col].apply(
        lambda d: json.dumps(
            {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in d.items()},
            sort_keys=True,
            ensure_ascii=False
        )
    )

# 2. Cast date-time fields safely
df['created_at'] = pd.to_datetime(df.get("created_at"), errors="coerce")
df['disclosed_at'] = pd.to_datetime(df.get("disclosed_at"), errors="coerce")

# 3. Select our raw landing columns
bronze_cols = [
    "id", "title", "created_at", "disclosed_at", "substate", "visibility",
    "has_bounty?", "vote_count", "original_report_id", "reporter_json",
    "team_json", "weakness_json", "structured_scope_json", "vulnerability_information"
]
bronze_df = df[[c for c in bronze_cols if c in df.columns]]

# 4. Save as a local compressed Parquet file
bronze_df.to_parquet("bronze_hackerone_reports.parquet", index=False)
print("✔ Bronze layer successfully written to 'bronze_hackerone_reports.parquet'!")

Processing nested fields and casting dates...
✔ Bronze layer successfully written to 'bronze_hackerone_reports.parquet'!


In [4]:
import duckdb
print("--- Phase 3: Transforming into Silver Layer ---")

# 1. Initialize a persistent local DuckDB database file
con = duckdb.connect(database="hackerone_pipeline.db")

# 2. Register the local parquet file as a table source inside DuckDB
con.execute("CREATE OR REPLACE TABLE bronze AS SELECT * FROM 'bronze_hackerone_reports.parquet';")
print("✔ Registered Bronze Parquet source in DuckDB.")

# 3. Create the core staging table with cleaned scalar attributes
con.sql("""
CREATE OR REPLACE TABLE stg_reports AS 
SELECT 
    CAST(id as BIGINT) as report_id, 
    title, 
    LOWER(NULLIF(substate,'')) as substate, -- Normalize text and handle blanks
    visibility, 
    "has_bounty?" as has_bounty, 
    CAST(vote_count AS INTEGER) as vote_count, 
    CAST(created_at AS TIMESTAMP) as created_at, 
    CAST(disclosed_at AS TIMESTAMP) as disclosed_at, 
    CAST(original_report_id AS BIGINT) as original_report_id, 
    reporter_json, 
    weakness_json, 
    team_json, 
    structured_scope_json, 
    vulnerability_information 
FROM bronze;
""")
print("✔ Base reports staging table created.")

# 4. Extract and flatten JSON arrays into typed staging entities
con.sql("""
CREATE OR REPLACE TABLE stg_reporter AS 
SELECT DISTINCT 
    reporter_json, 
    json_extract_string(reporter_json, '$.username') as username, 
    CAST(json_extract(reporter_json, '$.verified') AS BOOLEAN) as verified 
FROM bronze;
""")

con.sql("""
CREATE OR REPLACE TABLE stg_team AS 
SELECT DISTINCT 
    team_json, 
    json_extract_string(team_json, '$.handle') as handle, 
    CAST(json_extract(team_json, '$.id') AS BIGINT) as id, 
    CAST(json_extract(team_json,'$.offers_bounties') AS BOOLEAN) AS offers_bounties 
FROM bronze;
""")

con.sql("""
CREATE OR REPLACE TABLE stg_weakness AS 
SELECT DISTINCT 
    weakness_json, 
    json_extract_string(weakness_json,'$.name') AS weakness_name, 
    CAST(json_extract(weakness_json, '$.id') AS BIGINT) as id 
FROM bronze;
""")

con.sql("""
CREATE OR REPLACE TABLE stg_asset AS 
SELECT DISTINCT 
    structured_scope_json, 
    json_extract_string(structured_scope_json,'$.asset_identifier') AS asset_identifier, 
    json_extract_string(structured_scope_json,'$.asset_type') AS asset_type, 
    json_extract_string(structured_scope_json,'$.max_severity') AS max_severity 
FROM bronze;
""")

print("✔ Silver layer staging tables fully generated!")

# Quick check: See a sample of parsed teams
print("\nSample from stg_team:")
print(con.sql("SELECT handle, offers_bounties FROM stg_team LIMIT 5").df())

--- Phase 3: Transforming into Silver Layer ---
✔ Registered Bronze Parquet source in DuckDB.
✔ Base reports staging table created.
✔ Silver layer staging tables fully generated!

Sample from stg_team:
          handle  offers_bounties
0        algolia             True
1  rockstargames             True
2           qiwi             True
3            ibb             True
4        acronis             True


In [5]:
print("--- Phase 4: Creating Gold Layer (Star Schema) ---")

# 1. Build unique Dimension tables using deterministic surrogate keys from raw JSON
con.sql("""
CREATE OR REPLACE TABLE dim_reporter AS 
SELECT DISTINCT 
    hash(reporter_json) as reporter_key,
    username, 
    verified 
FROM stg_reporter;
""")

con.sql("""
CREATE OR REPLACE TABLE dim_team AS 
SELECT DISTINCT 
    hash(team_json) as team_key, 
    id as team_natural_id, 
    handle, 
    offers_bounties 
FROM stg_team;
""")

con.sql("""
CREATE OR REPLACE TABLE dim_weakness AS 
SELECT DISTINCT 
    hash(weakness_json) as weakness_key, 
    id as weakness_natural_id, 
    weakness_name 
FROM stg_weakness;
""")

con.sql("""
CREATE OR REPLACE TABLE dim_structured_scope AS 
SELECT DISTINCT 
    hash(structured_scope_json) as asset_key, 
    asset_identifier, 
    asset_type, 
    max_severity 
FROM stg_asset;
""")
print("✔ Dimension tables (Context) generated.")

# 2. Build the central Fact table, mapping rows back to dimensions via hashes
con.sql("""
CREATE OR REPLACE TABLE fact_report AS
SELECT 
    report_id,
    hash(reporter_json) as reporter_key,
    hash(team_json) as team_key,
    hash(weakness_json) as weakness_key,
    hash(structured_scope_json) as asset_key,
    substate,
    visibility,
    has_bounty,
    vote_count,
    created_at,
    disclosed_at,
    original_report_id
FROM stg_reports;
""")
print("✔ Fact table (Metrics) generated.")
print("✔ Gold Layer Star Schema completely structured!")

# Let's inspect the final Fact table structure
print("\nFact Table Preview:")
print(con.sql("SELECT report_id, reporter_key, team_key, vote_count FROM fact_report LIMIT 3").df())

--- Phase 4: Creating Gold Layer (Star Schema) ---
✔ Dimension tables (Context) generated.
✔ Fact table (Metrics) generated.
✔ Gold Layer Star Schema completely structured!

Fact Table Preview:
   report_id         reporter_key              team_key  vote_count
0     411337  3844753247148553500  16987200410034193644          35
1     311805  5584045288290698419   1145253177897127856           3
2     385322  9274002314680687824   2253045438949135403          24


In [6]:
print("--- Phase 5: Executing Quality Assurance (QA) Layer ---")

# 1. Row Loss Check: Compare Bronze raw records against Gold Fact table records
bronze_count = con.execute("SELECT COUNT(*) FROM bronze;").fetchone()[0]
fact_count = con.execute("SELECT COUNT(*) FROM fact_report;").fetchone()[0]

assert bronze_count == fact_count, f"❌ Data mismatch error! Bronze count ({bronze_count}) does not match Fact count ({fact_count})."
print(f"✔ Row Count Check Passed: Exactly {fact_count} records processed from Bronze to Gold.")

# 2. Referential Integrity Check: Ensure no orphaned team keys exist in the Fact table
orphaned_teams = con.sql("""
SELECT COUNT(*) FROM fact_report 
WHERE team_key NOT IN (SELECT team_key FROM dim_team);
""").fetchone()[0]

assert orphaned_teams == 0, f"❌ Critical FK issue: Found {orphaned_teams} records with missing dimension mappings!"
print("✔ Referential Integrity Passed: Zero orphaned dimension mappings detected.")

# 3. Completeness Profiling: Check how many records have missing target states
null_substates = con.execute("SELECT COUNT(*) FROM fact_report WHERE substate IS NULL;").fetchone()[0]
print(f"ℹ Data Profiling: Noted {null_substates} records with null/blank 'substate' classifications (this is expected raw behavior).")

print("\n⭐⭐ ALL QUALITY GATES PASSED! Data is certified clean. ⭐⭐")

--- Phase 5: Executing Quality Assurance (QA) Layer ---
✔ Row Count Check Passed: Exactly 12618 records processed from Bronze to Gold.
✔ Referential Integrity Passed: Zero orphaned dimension mappings detected.
ℹ Data Profiling: Noted 0 records with null/blank 'substate' classifications (this is expected raw behavior).

⭐⭐ ALL QUALITY GATES PASSED! Data is certified clean. ⭐⭐


In [7]:
print("--- Phase 6: Building Analytics Business Marts ---")

# Create a specialized reporting mart focusing on Vulnerability Classifications & Economics
con.sql("""
CREATE OR REPLACE TABLE mart_team_vulnerability_metrics AS
SELECT 
    t.handle as company_name,
    w.weakness_name as vulnerability_type,
    COUNT(f.report_id) as total_reports,
    SUM(CASE WHEN f.has_bounty THEN 1 ELSE 0 END) as rewarded_bounties,
    AVG(f.vote_count) as avg_community_votes
FROM fact_report f
JOIN dim_team t ON f.team_key = t.team_key
LEFT JOIN dim_weakness w ON f.weakness_key = w.weakness_key
GROUP BY company_name, vulnerability_type
HAVING total_reports >= 5
ORDER BY total_reports DESC;
""")
print("✔ Business analytics mart successfully materialized in DuckDB.")

# Convert the DuckDB table directly to a Pandas DataFrame for inspection
mart_df = con.table("mart_team_vulnerability_metrics").df()

print("\n--- Top 10 Rows in mart_team_vulnerability_metrics ---")
print(mart_df.head(10).to_string(index=False))

--- Phase 6: Building Analytics Business Marts ---
✔ Business analytics mart successfully materialized in DuckDB.

--- Top 10 Rows in mart_team_vulnerability_metrics ---
       company_name                     vulnerability_type  total_reports  rewarded_bounties  avg_community_votes
github-security-lab                                    NaN            212              212.0             9.858491
                ibb            Memory Corruption - Generic            196              182.0             3.852041
                ibb                                    NaN            146              140.0             5.760274
           security                 Information Disclosure            144               93.0            64.458333
      deptofdefense Cross-site Scripting (XSS) - Reflected            129                1.0             9.852713
             mailru                 Information Disclosure            108               58.0            14.666667
      deptofdefense             

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Phase 7: Generating Visual Insights ---")

# 1. Clean out the NaN values from the mart dataframe for a pristine visual
clean_mart = mart_df.dropna(subset=['vulnerability_type'])

# 2. Aggregate total reports by vulnerability type and grab the top 10
top_bugs = (
    clean_mart.groupby('vulnerability_type')['total_reports']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

# 3. Configure the plotting space
plt.figure(figsize=(11, 5))
sns.set_theme(style="whitegrid")

# 4. Draw a beautiful horizontal bar plot
sns.barplot(
    data=top_bugs,
    x='total_reports',
    y='vulnerability_type',
    palette='mako',
    hue='vulnerability_type',
    legend=False
)

# 5. Fine-tune typography and styling
plt.title('Top 10 Most Frequently Disclosed Vulnerabilities', fontsize=14, weight='bold', pad=15)
plt.xlabel('Total Aggregated Report Volume', fontsize=11, labelpad=10)
plt.ylabel('Vulnerability Category', fontsize=11)
plt.tight_layout()

# 6. Save the visualization directly to your project folder
plt.savefig("hackerone_vulnerability_trends.png", dpi=300)
print("✔ Chart successfully rendered and saved as 'hackerone_vulnerability_trends.png'!")

# 7. Safe Infrastructure Housekeeping
con.close()
print("✔ DuckDB database connection closed cleanly. Pipeline process complete!")